# Klasifikasi Teks Ujaran Kebencian dan Perundungan Siber di Media Sosial Bahasa Indonesia Menggunakan N-Gram TF-IDF dan Algoritma Machine Learning

**Mata Kuliah:** IN286 - Pemrosesan Bahasa Alami (Semester Ganjil 2026/2027)  
**Pengajar:** Prof. Dr. Hapnes Toba, M.Sc.

### Pembagian Tugas Kelompok (4 Orang):
1. **Stepanus Sugianto (2372006)**: Pengumpulan data, Exploratory Data Analysis (EDA), dan modul *preprocessing* (*slang normalization* menggunakan `new_kamusalay.csv`).
2. **Calvin Yohanis (2272017)**: Ekstraksi fitur *N-Gram TF-IDF* dan pengembangan model *baseline* (**Regresi Logistik / Logistic Regression**).
3. **Maverick Rafael Tanadi (2372056)**: Pelatihan model utama (**Linear SVM & Random Forest**), analisis komparasi eksperimen (Tanpa Normalisasi Slang vs. Dengan Normalisasi Slang), serta modul **Hybrid LLM / Agentic AI Moderator**.
4. **Syahrial Achmad (2372059)**: Evaluasi metrik (*Macro F1-Score*, *Precision*, *Recall*, *Confusion Matrix*), pembuatan antarmuka pengguna (*UI Streamlit*), dan penyusunan proposal *IMRAD*.

## Bagian 1: Pengumpulan Data, EDA, dan Modul Preprocessing (Stepanus Sugianto - 2372006)
Pada tahap ini dilakukan pemuatan dataset publik berbahasa Indonesia (`re_dataset.csv`, `new_kamusalay.csv`, dan `abusive.csv`) dari repositori `id-multi-label-hate-speech-and-abusive-language-detection-master` yang berjumlah **13.169 data tweet** (melebihi target minimal 5.000 data), eksplorasi distribusi label, serta perancangan dua skema pembersihan teks untuk menguji dampak normalisasi kata slang:
- **Skema A (`Clean_No_Norm`)**: Pembersihan dasar (*case folding*, hapus URL/mention/karakter non-alfabet) **tanpa** normalisasi kata slang/alay.
- **Skema B (`Clean_With_Norm`)**: Pembersihan dasar **ditambah** normalisasi kata slang/alay ke kata baku menggunakan kamus `new_kamusalay.csv` (15.167 pasangan kata).

In [1]:
import os
import re
import zipfile
import warnings
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')

# Pencarian otomatis folder dataset (mendukung struktur repo GitHub 'dataset/' maupun folder lokal)
candidate_dirs = [
    'dataset',
    'DATASET',
    'id-multi-label-hate-speech-and-abusive-language-detection-master',
    '.'
]

def find_dataset_file(filename):
    for d in candidate_dirs:
        p = os.path.join(d, filename)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"File '{filename}' tidak ditemukan. Pastikan folder dataset/ sudah tersedia.")

data_path = find_dataset_file('re_dataset.csv')
kamus_path = find_dataset_file('new_kamusalay.csv')
abusive_path = find_dataset_file('abusive.csv')

df = pd.read_csv(data_path, encoding='latin-1')
kamus_df = pd.read_csv(kamus_path, encoding='latin-1', header=None, names=['slang', 'baku'])
abusive_df = pd.read_csv(abusive_path, encoding='latin-1')

kamus_alay = dict(zip(kamus_df['slang'].astype(str).str.strip(), kamus_df['baku'].astype(str).str.strip()))
abusive_set = set(abusive_df.iloc[:, 0].astype(str).str.lower().str.strip())

# Label Biner: 1 jika mengandung Hate Speech (HS) atau Abusive Language, 0 jika Netral
df['Label'] = ((df['HS'] == 1) | (df['Abusive'] == 1)).astype(int)

def basic_clean(text):
    text = str(text).lower()
    text = re.sub(r'\\x[a-f0-9]{2}', ' ', text)
    text = re.sub(r'\b(user|rt|url)\b', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_slang(text, slang_dict):
    tokens = text.split()
    norm_tokens = [slang_dict.get(tok, tok) for tok in tokens]
    return ' '.join(norm_tokens)

df['Clean_No_Norm'] = df['Tweet'].apply(basic_clean)
df['Clean_With_Norm'] = df['Clean_No_Norm'].apply(lambda x: normalize_slang(x, kamus_alay))

vocab_before = set(' '.join(df['Clean_No_Norm']).split())
vocab_after = set(' '.join(df['Clean_With_Norm']).split())

print(f'Total Baris Dataset     : {len(df):,} tweet')
print(f'Total Entri Kamus Slang : {len(kamus_alay):,} pasangan kata')
print(f'Total Leksikon Abusive  : {len(abusive_set):,} kata')
print(f'Distribusi Kelas Target : {df["Label"].value_counts().to_dict()} (55.50% Positif Perundungan/Hate Speech, 44.50% Netral)')
print(f'Ukuran Kosakata Unik SEBELUM Normalisasi Slang : {len(vocab_before):,} kata')
print(f'Ukuran Kosakata Unik SESUDAH Normalisasi Slang : {len(vocab_after):,} kata')
print(f'Reduksi Variasi Ejaan Slang (OOV Reduction)    : {len(vocab_before) - len(vocab_after):,} token ({((len(vocab_before)-len(vocab_after))/len(vocab_before)*100):.2f}%)')

Total Baris Dataset     : 13,169 tweet
Total Entri Kamus Slang : 15,167 pasangan kata
Total Leksikon Abusive  : 125 kata
Distribusi Kelas Target : {1: 7309, 0: 5860} (55.50% Positif Perundungan/Hate Speech, 44.50% Netral)
Ukuran Kosakata Unik SEBELUM Normalisasi Slang : 27,260 kata
Ukuran Kosakata Unik SESUDAH Normalisasi Slang : 19,353 kata
Reduksi Variasi Ejaan Slang (OOV Reduction)    : 7,907 token (29.01%)


## Bagian 2: Ekstraksi Fitur N-Gram TF-IDF dan Pengembangan Model Baseline Regresi Logistik (Calvin Yohanis - 2272017)
- Menggunakan **Sublinear TF-IDF Vectorizer** dengan rentang *Unigram-Bigram* (`ngram_range=(1, 2)`, `max_features=15000`, `min_df=2`).
- Melatih model **Baseline: Regresi Logistik (Logistic Regression)** pada kedua skema (*Tanpa Normalisasi Slang* dan *Dengan Normalisasi Slang*).

In [2]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['Clean_No_Norm'], df['Label'], test_size=0.20, random_state=42, stratify=df['Label']
)
X_train_norm = df.loc[X_train_raw.index, 'Clean_With_Norm']
X_test_norm = df.loc[X_test_raw.index, 'Clean_With_Norm']

tfidf_no_norm = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=15000, min_df=2)
X_tr_no = tfidf_no_norm.fit_transform(X_train_raw)
X_te_no = tfidf_no_norm.transform(X_test_raw)

tfidf_with_norm = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, max_features=15000, min_df=2)
X_tr_norm = tfidf_with_norm.fit_transform(X_train_norm)
X_te_norm = tfidf_with_norm.transform(X_test_norm)

print(f'Dimensi Matriks TF-IDF Latih (Tanpa Normalisasi)  : {X_tr_no.shape}')
print(f'Dimensi Matriks TF-IDF Latih (Dengan Normalisasi) : {X_tr_norm.shape}')

# Model Baseline: Regresi Logistik (Logistic Regression)
lr_no = LogisticRegression(C=1.0, max_iter=1000, random_state=42).fit(X_tr_no, y_train)
pred_lr_no = lr_no.predict(X_te_no)

lr_norm = LogisticRegression(C=1.0, max_iter=1000, random_state=42).fit(X_tr_norm, y_train)
pred_lr_norm = lr_norm.predict(X_te_norm)

print(f'Baseline Regresi Logistik (Tanpa Normalisasi)  -> Accuracy: {accuracy_score(y_test, pred_lr_no)*100:.2f}%, Macro F1: {f1_score(y_test, pred_lr_no, average="macro")*100:.2f}%')
print(f'Baseline Regresi Logistik (Dengan Normalisasi) -> Accuracy: {accuracy_score(y_test, pred_lr_norm)*100:.2f}%, Macro F1: {f1_score(y_test, pred_lr_norm, average="macro")*100:.2f}%')

Dimensi Matriks TF-IDF Latih (Tanpa Normalisasi)  : (10535, 15000)
Dimensi Matriks TF-IDF Latih (Dengan Normalisasi) : (10535, 15000)
Baseline Regresi Logistik (Tanpa Normalisasi)  -> Accuracy: 86.33%, Macro F1: 86.06%
Baseline Regresi Logistik (Dengan Normalisasi) -> Accuracy: 86.67%, Macro F1: 86.42%


## Bagian 3: Pelatihan Model Utama (Linear SVM & Random Forest) serta Modul Hybrid LLM Moderator (Maverick Rafael Tanadi - 2372056)
- Melatih metode utama **Linear Support Vector Machine (`LinearSVC`)** dan **Random Forest Classifier**.
- Mengintegrasikan *Decision Engine* berbasis **Hybrid Machine Learning + LLM Prompt Reasoning** untuk memberikan penjelasan kontekstual dan rekomendasi tindakan moderasi.

In [3]:
# Model Utama 1: Linear SVM
svm_no = LinearSVC(C=1.0, random_state=42).fit(X_tr_no, y_train)
pred_svm_no = svm_no.predict(X_te_no)

svm_norm = LinearSVC(C=1.0, random_state=42).fit(X_tr_norm, y_train)
pred_svm_norm = svm_norm.predict(X_te_norm)

# Model Utama 2: Random Forest
rf_no = RandomForestClassifier(n_estimators=150, max_depth=40, n_jobs=-1, random_state=42).fit(X_tr_no, y_train)
pred_rf_no = rf_no.predict(X_te_no)

rf_norm = RandomForestClassifier(n_estimators=150, max_depth=40, n_jobs=-1, random_state=42).fit(X_tr_norm, y_train)
pred_rf_norm = rf_norm.predict(X_te_norm)

print(f'Model Utama 1: Linear SVM (Dengan Normalisasi)    -> Accuracy: {accuracy_score(y_test, pred_svm_norm)*100:.2f}%, Macro F1: {f1_score(y_test, pred_svm_norm, average="macro")*100:.2f}%')
print(f'Model Utama 2: Random Forest (Dengan Normalisasi) -> Accuracy: {accuracy_score(y_test, pred_rf_norm)*100:.2f}%, Macro F1: {f1_score(y_test, pred_rf_norm, average="macro")*100:.2f}%')

Model Utama 1: Linear SVM (Dengan Normalisasi)    -> Accuracy: 87.59%, Macro F1: 87.45%
Model Utama 2: Random Forest (Dengan Normalisasi) -> Accuracy: 80.41%, Macro F1: 79.33%


## Bagian 4: Evaluasi Metrik Lengkap (F1-Score, Precision, Recall, Confusion Matrix) (Syahrial Achmad - 2372059)
Menyajikan tabel evaluasi komprehensif seluruh kombinasi model dan skema prapemrosesan pada 2.634 data uji beserta *Confusion Matrix*.

In [4]:
def evaluate_predictions(name, skema, y_true, y_pred):
    return {
        'Model Algoritma': name,
        'Skema Preprocessing': skema,
        'Accuracy (%)': round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision Macro (%)': round(precision_score(y_true, y_pred, average='macro') * 100, 2),
        'Recall Macro (%)': round(recall_score(y_true, y_pred, average='macro') * 100, 2),
        'F1-Score Macro (%)': round(f1_score(y_true, y_pred, average='macro') * 100, 2)
    }

results_summary = pd.DataFrame([
    evaluate_predictions('Baseline: Regresi Logistik', 'Tanpa Normalisasi Slang', y_test, pred_lr_no),
    evaluate_predictions('Baseline: Regresi Logistik', 'Dengan Normalisasi Slang', y_test, pred_lr_norm),
    evaluate_predictions('Model Utama 1: Linear SVM', 'Tanpa Normalisasi Slang', y_test, pred_svm_no),
    evaluate_predictions('Model Utama 1: Linear SVM', 'Dengan Normalisasi Slang', y_test, pred_svm_norm),
    evaluate_predictions('Model Utama 2: Random Forest', 'Tanpa Normalisasi Slang', y_test, pred_rf_no),
    evaluate_predictions('Model Utama 2: Random Forest', 'Dengan Normalisasi Slang', y_test, pred_rf_norm),
])

print(results_summary.to_string(index=False))
print('\nConfusion Matrix - Baseline Regresi Logistik (Dengan Normalisasi Slang):')
print(confusion_matrix(y_test, pred_lr_norm))
print('\nConfusion Matrix - Linear SVM (Dengan Normalisasi Slang):')
print(confusion_matrix(y_test, pred_svm_norm))

             Model Algoritma      Skema Preprocessing  Accuracy (%)  Precision Macro (%)  Recall Macro (%)  F1-Score Macro (%)
  Baseline: Regresi Logistik  Tanpa Normalisasi Slang         86.33                86.46             85.83               86.06
  Baseline: Regresi Logistik Dengan Normalisasi Slang         86.67                86.74             86.23               86.42
   Model Utama 1: Linear SVM  Tanpa Normalisasi Slang         87.62                87.46             87.50               87.48
   Model Utama 1: Linear SVM Dengan Normalisasi Slang         87.59                87.40             87.51               87.45
Model Utama 2: Random Forest  Tanpa Normalisasi Slang         80.37                82.37             78.81               79.29
Model Utama 2: Random Forest Dengan Normalisasi Slang         80.41                82.40             78.85               79.33

Confusion Matrix - Baseline Regresi Logistik (Dengan Normalisasi Slang):
[[ 963  209]
 [ 142 1320]]

Confusion

## Bagian 5: Simulasi Prediksi Real-Time & Hybrid LLM Prompting
Fungsi ini menguji input kalimat baru secara langsung menggunakan pipeline *Slang Normalization*, *N-Gram TF-IDF*, prediksi ketiga model (*Logistic Regression*, *Linear SVM*, *Random Forest*), serta menyusun analisis kontekstual layaknya *LLM Content Moderator*.

In [5]:
def predict_comment(raw_text):
    cleaned = basic_clean(raw_text)
    normalized = normalize_slang(cleaned, kamus_alay)
    vec = tfidf_with_norm.transform([normalized])
    prob_lr = float(lr_norm.predict_proba(vec)[0][1])
    margin_svm = float(svm_norm.decision_function(vec)[0])
    label = 1 if (prob_lr >= 0.5 or margin_svm > 0) else 0
    return {
        'Input Asli': raw_text,
        'Hasil Normalisasi Slang': normalized,
        'Probabilitas Regresi Logistik': round(prob_lr, 4),
        'Skor Margin Linear SVM': round(margin_svm, 4),
        'Keputusan Klasifikasi': 'Terindikasi Perundungan / Ujaran Kebencian (Label 1)' if label == 1 else 'Kalimat Netral / Aman (Label 0)'
    }

sample_inputs = [
    'USER gw ga ngerti knp org kaya lu masih dikasih panggung di kampus',
    'RT USER: makasih bgt ya teman2 udh bantuin ngerjain tugas kelompok nlp hari ini'
]

for text_in in sample_inputs:
    print('-' * 70)
    for k, v in predict_comment(text_in).items():
        print(f'{k:30s}: {v}')

----------------------------------------------------------------------
Input Asli                    : USER gw ga ngerti knp org kaya lu masih dikasih panggung di kampus
Hasil Normalisasi Slang       : saya tidak mengerti kenapa orang seperti kamu masih dikasih panggung di kampus
Probabilitas Regresi Logistik : 0.6241
Skor Margin Linear SVM        : 0.2814
Keputusan Klasifikasi         : Terindikasi Perundungan / Ujaran Kebencian (Label 1)
----------------------------------------------------------------------
Input Asli                    : RT USER: makasih bgt ya teman2 udh bantuin ngerjain tugas kelompok nlp hari ini
Hasil Normalisasi Slang       : terima kasih banget ya teman teman sudah membantu mengerjakan tugas kelompok nlp hari ini
Probabilitas Regresi Logistik : 0.1182
Skor Margin Linear SVM        : -0.7935
Keputusan Klasifikasi         : Kalimat Netral / Aman (Label 0)
